# 04c. 유전자 줄이기 실험 (E 조합 + chi2)

- 담당: **④ 경량화·통합** (이 파일은 담당자만 수정)
- 목적: E 조합(0/1 + 개수 + 변이 종류 + 아미노산 쌍 + 끊김 표시, 9,174열)에서 chi2로 열을 줄여도 성능이 유지되는지 확인
- 모델: 로지스틱 회귀 (C=0.1, class_weight=balanced, max_iter=3000), 팀 그룹 K-fold
- 누수 방지: chi2 선택은 Pipeline 안에서 fold의 학습 부분에서만 계산해요. E 피처 계산은 한 환자 안에서만 하는 계산이라 미리 한 번 계산해 둬요.
- 기록 파일: `experiments/log_P_light.csv` (`1109_E_chi2_k○○_gkf`)
- 예측 파일: `experiments/preds/P_light__logreg_E_chi2_k500_1109__*`
- ⚠️ Run All 하면 "3. 열 수별 교차검증"이 다시 기록돼 기록표에 같은 줄이 또 생겨요.

In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression

from src.data_loader import load_data, split_xy, encode_target
from src.preprocessing import binarize_mutations
from src.features import (mutation_count, mutation_type_counts,
                          amino_acid_pair_counts, gene_truncation_flags)
from src.validation import get_folds, run_cv
from src.evaluation import log_result, save_predictions

train, test = load_data(DATA_DIR)   # test는 마지막 예측에만 사용
X, y_raw = split_xy(train)
y, le = encode_target(y_raw)
folds = get_folds(y)                # 팀 공통 그룹 K-fold
print("train", train.shape, "| 암종 수", len(le.classes_), "| fold 수", len(folds))

## 2. E 피처 미리 계산
② 02_features의 make_features_E와 같은 계산이에요. 한 환자 안에서만 하는 계산이라(통계·선택 없음) 미리 한 번 계산해도 누수가 없어요.

In [ ]:
def make_features_E(X):
    """② E 조합 피처 (02_features의 make_features_E와 같은 계산). 한 환자 안에서만 계산."""
    return pd.concat([binarize_mutations(X),
                      np.log1p(mutation_count(X)),
                      np.log1p(mutation_type_counts(X)),
                      np.log1p(amino_acid_pair_counts(X)),
                      gene_truncation_flags(X)], axis=1)

X_E = make_features_E(X).astype(np.float32)   # 약 1분
print("E 피처:", X_E.shape)

In [1]:
## 3. 열 수별 교차검증 — 2000 / 1000 / 500 / 300 / 100
chi2로 열을 k개만 골라 같은 로지스틱 회귀로 비교해요. 고르는 기준은 fold의 학습 부분에서만 계산해요.

SyntaxError: invalid syntax (3991529215.py, line 2)